# Semana 3: masa invariante de dimuones (`02_masa_invariante.ipynb`)

Programa: [`docs/programa_mes1.md`](../../docs/programa_mes1.md), sección *Semana 3*.

Objetivo: el espectro μ⁺μ⁻ de 0.25 a 150 GeV en escala log-log, con las resonancias identificadas y la posición
de tres picos medida. Necesitas haber terminado E2.7 (`leer_muones` y `hist_cms`).

**Cómo trabajar este notebook.** Completa las celdas marcadas con `# TODO`. Las celdas de **verificación**
(las que terminan en `assert`) ya están escritas: no las borres ni cambies las tolerancias. Si una verificación
falla, intenta corregirlo tú; si sigue fallando, anótalo en `bitacora.md` en *qué no entendí*. Las celdas
*Tu respuesta* se contestan con tus palabras (2 a 5 frases) y se discuten en la reunión.

Antes del commit: *Kernel → Restart & Run All* (todo debe correr de principio a fin) y luego borra las salidas.

In [ ]:
# --- Configuración: funciona igual en Google Colab y en un entorno conda local ---
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RREYESAL/Hands_on_SSIPN"  # <- si trabajas en un fork, pon aquí su URL
DIR_COLAB = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "uproot", "awkward", "vector", "hist", "mplhep", "pyyaml", "aiohttp", "requests", "psutil"], check=True)
    if not DIR_COLAB.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(DIR_COLAB)], check=True)
    os.chdir(DIR_COLAB / "notebooks" / "mes1")

# Buscar la raíz del repositorio (la carpeta que contiene src/hepss) y hacer importable hepss
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hepss").is_dir()), None)
if RAIZ is None:
    raise RuntimeError("No encuentro src/hepss: corre el notebook desde dentro del repositorio")
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))
os.chdir(RAIZ)  # rutas relativas (configs/, data/, plots/) respecto a la raíz
Path("plots").mkdir(exist_ok=True)

# Recarga automática de hepss: si editas src/hepss/*.py, no hace falta reiniciar el kernel
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

# Para forzar los datos sintéticos (sin red), descomentar:
# os.environ["HEPSS_SINTETICO"] = "1"
print("Colab:", EN_COLAB, "| raíz del repositorio:", RAIZ)

### Datos: CMS Open Data o respaldo sintético

`hepss.io.abrir_events` intenta leer CMS Open Data por HTTPS (y por XRootD si `fsspec-xrootd` está instalado).
Si no lo consigue en 30 s (sin red, servidor caído, firewall) o si la ruta no está configurada, **genera una vez
un archivo sintético en `data/` con las mismas ramas** y lo usa, avisando con un `UserWarning`. La variable
`ORIGEN` vale `"opendata"` o `"sintetico"`. Usa siempre `etiqueta_cms(ax, ORIGEN)` en tus gráficas: con datos
sintéticos pone *Simulación sintética* en lugar de *CMS Open Data*, para que nunca se confundan.

Las verificaciones están escritas para pasar con ambos orígenes. Las que solo tienen sentido con datos reales
se comprueban solo si `ORIGEN == "opendata"`.

In [ ]:
import numpy as np
import awkward as ak
import hist
import vector
import matplotlib.pyplot as plt
import mplhep as hep

from hepss.io import leer_muones, zip_muones
from hepss.graficas import hist_cms, etiqueta_cms
from hepss.constantes import M_MU

vector.register_awkward()
eventos, ORIGEN = leer_muones("dimuon2012", n_eventos=2_000_000)
muones = zip_muones(eventos)          # muones.pt, muones.eta, ... en lugar de eventos.Muon_pt
print(ORIGEN, len(eventos), "eventos")

## Física: masa invariante

$$ m_{\mu\mu}^2 = (E_1 + E_2)^2 - |\vec p_1 + \vec p_2|^2 \approx 2\, p_{T1}\, p_{T2}\, [\cosh(\eta_1 - \eta_2) - \cos(\phi_1 - \phi_2)] $$

La aproximación de la derecha desprecia la masa del muon. Antes de seguir, deriva la aproximación a mano
(escribe E, pz y pT en función de pT, η, φ) y anota en la bitácora qué paso usa que la masa es despreciable.

## E3.1 Pares μ⁺μ⁻: exactamente dos muones de carga opuesta

Construye las máscaras `dos` (exactamente 2 muones) y `opuesta` (carga opuesta), la selección `sel`, los eventos
`pares = muones[sel]` y los muones `mu1`, `mu2`. Imprime la fracción de eventos que pasa cada paso.

In [ ]:
# TODO
dos = ...
opuesta = ...
sel = ...
pares = ...
mu1, mu2 = ..., ...

In [ ]:
# Verificación E3.1
assert ak.all(mu1.charge + mu2.charge == 0)
assert ak.all(ak.num(pares.pt) == 2)

¿La condición `suma de cargas == 0` es equivalente a "carga opuesta" para cualquier número de muones? ¿Qué harías con los eventos de 3 o más muones?

**Tu respuesta:** _(escribe aquí)_

## E3.2 Masa con la fórmula sin masa

Implementa `masa_sin_masa` en `src/hepss/cinematica.py` (sin bucles: con funciones de numpy funciona igual para
awkward) y calcula `m_aprox` para todos los pares.

In [ ]:
from hepss.cinematica import masa_sin_masa

# TODO
m_aprox = ...

In [ ]:
# Verificación E3.2: pT = (10, 10) GeV, Δη = 0, Δφ = π
m_prueba = masa_sin_masa(10.0, 0.5, 0.0, 10.0, 0.5, np.pi)
print("m(prueba) =", m_prueba, "GeV")
assert abs(m_prueba - 20.0) < 1e-9
assert len(m_aprox) == len(mu1)

## E3.3 Masa con `vector`

Construye los cuadrivectores de `mu1` y `mu2` con `pt`, `eta`, `phi` y `mass` (puedes implementar
`hepss.cinematica.cuadrivectores`) y calcula `m_vec` como la masa de la suma. Calcula la diferencia relativa
`dif_rel = (m_aprox - m_vec) / m_vec` y, para los pares con `m_vec > 5`, guarda su valor absoluto en `d_alta`
(arreglo de numpy). Revisa qué valor tiene `Muon_mass` en el archivo y en qué unidades.

In [ ]:
# TODO
m_vec = ...
dif_rel = ...
d_alta = ...

In [ ]:
# Verificación E3.3
print(f"Para m > 5 GeV: mediana {np.median(d_alta):.1e}, percentil 99 {np.quantile(d_alta, 0.99):.1e}, máximo {d_alta.max():.1e}")
assert np.quantile(d_alta, 0.99) < 1e-3
assert np.median(d_alta) > 0, "Si la diferencia es exactamente 0, revisa qué masa le pasas a vector"

¿Por qué la diferencia no es menor que 0.1 % para **todos** los pares con mμμ > 5 GeV? ¿Qué pares se desvían más?

**Tu respuesta:** _(escribe aquí)_

## E3.4 Diferencia relativa en función de mμμ

Grafica la mediana de `-dif_rel` en bins logarítmicos de masa entre 0.25 y 150 GeV (escala log en ambos ejes), con una
línea vertical en 2mμ. Guarda los centros de los bins en `centros` y las medianas (de `dif_rel`, con su signo) en
`mediana`, ambos arreglos de numpy (usa `np.nan` en bins con pocos pares).

In [ ]:
# TODO
centros = ...
mediana = ...

In [ ]:
# Verificación E3.4: la diferencia es mucho mayor a baja masa que a 10 GeV
assert -mediana[1] > 10 * -mediana[np.nanargmin(np.abs(centros - 10))]

¿Por qué la diferencia crece cerca de 2mμ = 0.211 GeV? ¿Qué signo tiene y por qué? Intenta estimar su orden de magnitud en función de mμ/mμμ.

**Tu respuesta:** _(escribe aquí)_

## E3.5 Espectro μ⁺μ⁻ en escala log-log

Histograma `h_m` de `m_vec` con eje `eje_m = hist.axis.Regular(500, 0.25, 150, transform=hist.axis.transform.log, ...)`,
graficado con `hist_cms(..., logx=True, logy=True)` y guardado en `plots/espectro_mumu.png`.

In [ ]:
# TODO
eje_m = ...
h_m = ...

La celda siguiente es una **ayuda** (no un ejercicio): cuenta los bins que sobresalen respecto a una línea base suave. También puedes contarlos a ojo.

In [ ]:
# Verificación E3.5
from scipy.ndimage import median_filter, uniform_filter1d
y = h_m.values()
base = median_filter(y, size=41, mode="nearest")
razon = uniform_filter1d(y, 3) / np.maximum(base, 1)
en_pico = razon > 1.3
inicios = np.flatnonzero(en_pico & ~np.r_[False, en_pico[:-1]])
finales = np.flatnonzero(en_pico & ~np.r_[en_pico[1:], False])
picos = [eje_m.centers[a:b + 1][np.argmax(y[a:b + 1])] for a, b in zip(inicios, finales)]
print("Picos encontrados (GeV):", np.round(picos, 3))
assert len(picos) >= 7, f"Solo {len(picos)} picos"

¿Por qué bins logarítmicos y no de ancho constante? ¿Por qué escala log en el eje y?

**Tu respuesta:** _(escribe aquí)_

## E3.6 Resonancias marcadas

Completa el diccionario `RESONANCIAS` (nombre → masa en **GeV** del PDG) con η, ρ/ω, φ, J/ψ, ψ(2S), Υ(1S), Υ(2S),
Υ(3S) y Z, y marca cada una sobre el espectro con `ax.annotate`. Guarda la figura en `plots/espectro_mumu_etiquetas.png`.

In [ ]:
# TODO: masas del PDG en GeV (el PDG las da en MeV)
RESONANCIAS = {
    r"$J/\psi$": ...,
    # ...
}

In [ ]:
# Verificación E3.6: cada etiqueta cae sobre un pico
assert len(RESONANCIAS) >= 9
for nombre, masa in RESONANCIAS.items():
    i = eje_m.index(masa)
    r = razon[max(i - 2, 0):i + 3].max()
    print(f"{nombre:18s} m = {masa:8.4f} GeV   razón máxima = {r:.2f}")
    assert r > 1.15, f"No hay pico en {nombre}"

Para cada resonancia, ¿cuál es su contenido de quarks? ¿Por qué el ρ y el ω no se pueden separar?

**Tu respuesta:** _(escribe aquí)_

## E3.7 Posición de los picos

Mide la posición del pico de J/ψ, Υ(1S) y Z como el bin máximo en una ventana fina alrededor de cada uno.
Elige tú la ventana y el ancho de bin; justifica la elección. Guarda los resultados en el diccionario `posiciones`.
Pista: piensa en cómo se compara el ancho de bin con la tolerancia, y en cuánto fluctúa el contenido de un bin.

In [ ]:
# TODO
posiciones = {
    "J/psi": ...,
    "Upsilon(1S)": ...,
    "Z": ...,
}

In [ ]:
# Verificación E3.7
esperado = {"J/psi": (3.097, 0.01), "Upsilon(1S)": (9.46, 0.05), "Z": (91.2, 0.5)}
for nombre, (valor, tol) in esperado.items():
    print(f"{nombre:12s} {posiciones[nombre]:.3f} GeV (esperado {valor} ± {tol})")
    assert abs(posiciones[nombre] - valor) < tol, nombre

¿Qué ancho de bin usaste y por qué? En datos reales el máximo del Z sale un poco por debajo de 91.19 GeV: ¿por qué?

**Tu respuesta:** _(escribe aquí)_

## E3.8 Resolución de masa del J/ψ: barril contra endcap

Repite el espectro del J/ψ (2.8 a 3.4 GeV) separando los pares con ambos muones en el barril y con ambos en el endcap
(los límites están en `configs/seleccion_z.yaml`, sección `regiones_eta`). Estima la anchura de cada pico, `s_b` y `s_e`
en GeV, con el método que elijas (por ejemplo la anchura a media altura tras restar el fondo, con FWHM = 2.355 σ para
una gaussiana). Grafica ambos picos normalizados en `plots/jpsi_barril_endcap.png`.

In [ ]:
import yaml
regiones = yaml.safe_load(open("configs/seleccion_z.yaml"))["regiones_eta"]

# TODO
s_b = ...
s_e = ...

In [ ]:
# Verificación E3.8
print(f"σ(barril) = {1000*s_b:.1f} MeV, σ(endcap) = {1000*s_e:.1f} MeV, cociente = {s_e/s_b:.2f}")
assert s_e > s_b
assert 0.005 < s_b < 0.2, "Revisa unidades y método"

La anchura natural del J/ψ es de 93 keV. ¿Qué estás midiendo entonces? ¿Por qué la anchura es mayor en el endcap?

**Tu respuesta:** _(escribe aquí)_

## Preguntas para la bitácora

1. ¿Por qué el fondo cae con la masa?
2. ¿Por qué el Υ(1S), Υ(2S) y Υ(3S) se ven como picos separados y no un solo bulto?

**Tu respuesta:** _(escribe aquí)_